# Higgs Physics and the HIGGS Dataset: A Beginner Guide

**Author:** Saumya Mishra (documentation lead) | **Project:** Higgs Event Lab | **Contains no model results.**

This notebook explains, in plain language, three things: what the Higgs field and the Higgs boson are, what the UCI HIGGS dataset contains, and what our project does and does not claim. It is the teaching companion to `docs/08_PHYSICS_BACKGROUND.md` and `docs/09_DATASET_GUIDE.md`.

**How each section is organised:** the question it answers, why it matters, what any code does, what output to expect, and how to interpret it.

> **Scientific boundary.** We classify *simulated* events labelled signal or background. We do not simulate the Higgs field, claim a discovery, produce detector-grade results, or turn classifier scores into real-world probabilities.

## 1. What is the Higgs field?

**Question:** why do we talk about a "field" at all, and what is special about the Higgs field?

**Why it matters:** the whole task makes sense only if you know why physicists care about the Higgs.

**Explanation:** in modern physics each kind of particle is a ripple in a field that fills all of space. Most fields are zero in empty space. The Higgs field is different because its lowest-energy value is *not* zero, so it is present everywhere. Particles that interact with it behave as if they have mass, and a stronger interaction means a larger mass.

**Interpretation:** the "crowd in a room" picture is only an analogy. Nothing drags the particle; the interaction is described by quantum field theory.

## 2. The Higgs mechanism

**Question:** what problem does the Higgs mechanism solve?

**Why it matters:** it explains why the W and Z bosons are heavy even though the symmetry of the electroweak theory seems to forbid a mass term.

**Explanation:** the laws stay symmetric, but the Higgs field settles into a lowest-energy state that is not symmetric, often drawn as a ball rolling off the top of a Mexican hat into the valley. This is called spontaneous symmetry breaking. As a result the W and Z bosons acquire mass, while the photon stays massless. Quarks and charged leptons obtain mass through separate couplings to the same field.

**Common mistakes to avoid:** the Higgs field does not give mass to everything (most of a proton's mass comes from the strong force), and the boson is not the thing that gives mass. The field does that.

## 3. The Higgs boson and its discovery

**Question:** what is the Higgs boson, and how was it found?

**Why it matters:** it separates the real 2012 discovery from the simulated benchmark in this project.

**Explanation:** the Higgs boson is a quantum ripple of the Higgs field, a neutral particle with no spin and a mass of about 125 GeV. It decays almost immediately, so detectors see only its decay products. In July 2012 the ATLAS and CMS experiments at CERN announced the observation of a new particle consistent with it, and the 2013 Nobel Prize in Physics went to Englert and Higgs. Most collisions at the LHC are ordinary background, so picking out interesting events is a classification problem. That is where machine learning enters.

## 4. What is in the HIGGS dataset?

**Question:** what exactly are the "signal" and "background" in the dataset?

**Why it matters:** the original paper (Baldi, Sadowski and Whiteson, 2014) describes the signal as a process producing *new, hypothetical (exotic) Higgs bosons*, while the background is a top-quark process with identical decay products but different kinematics. So the dataset is a benchmark, not a rerun of the 2012 discovery. All events are computer-simulated.

**What the next code cell does:** it reads the project feature dictionary, `docs/feature_dictionary.csv`, with the standard library only, and counts how many columns belong to each group.

**Expected output:** one label column, 21 low-level features and 7 high-level features, 29 columns in total.

**Interpretation:** if the counts differ, the dictionary no longer matches the UCI schema and the unit test `test_feature_dictionary.py` should also fail.

In [1]:
import csv
from collections import Counter
from pathlib import Path

dictionary_path = Path("..") / ".." / "docs" / "feature_dictionary.csv"
with open(dictionary_path, newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

print("Total columns in the file:", len(rows))
for group, count in Counter(r["group"] for r in rows).items():
    print(f"{group:<11} {count}")

Total columns in the file: 29
target      1
low_level   21
high_level  7


## 5. The two feature groups

**Question:** how do low-level and high-level features differ?

**Why it matters:** the course project asks whether physicist-designed features still help once a model can learn from the raw measurements.

**Explanation:** the 21 low-level features are direct measurements: the momentum (pT) and angles (eta, phi) of the lepton and up to four jets, a b-tag value for each jet, and the missing energy that stands in for the invisible neutrino. The 7 high-level features are derived from them and are invariant masses of particle combinations, such as the two b-jets or the full W W b b system. Invariant masses are useful because a heavy parent particle shows up as a bump in the mass of its decay products.

**What the next code cell does:** it prints the feature names of each group in file order.

**Expected output:** the lepton and missing-energy features first, then four jets of four features each, then the seven masses.

In [2]:
low = [r["name"] for r in rows if r["group"] == "low_level"]
high = [r["name"] for r in rows if r["group"] == "high_level"]

print("Low-level (21):")
print(", ".join(low))
print()
print("High-level (7):")
print(", ".join(high))

Low-level (21):
lepton_pt, lepton_eta, lepton_phi, missing_energy_magnitude, missing_energy_phi, jet_1_pt, jet_1_eta, jet_1_phi, jet_1_btag, jet_2_pt, jet_2_eta, jet_2_phi, jet_2_btag, jet_3_pt, jet_3_eta, jet_3_phi, jet_3_btag, jet_4_pt, jet_4_eta, jet_4_phi, jet_4_btag

High-level (7):
m_jj, m_jjj, m_lv, m_jlv, m_bb, m_wbb, m_wwbb


## 6. Pitfalls when reading the file

**Question:** what silent mistakes can ruin the experiment before any model is trained?

**Explanation:**

1. **Off-by-one columns.** UCI counts the low-level features as columns 2 to 22 (starting from 1). Python starts at 0, so the label is column 0 and the features are columns 1 to 28.
2. **Label leakage.** The label (1 for signal, 0 for background) must never appear in the feature matrix or in a scaler.
3. **Fitting on the test set.** Split first, then fit scalers on the training part only.
4. **Angles are periodic.** The angles `phi` wrap around, so minus pi and plus pi are the same direction.
5. **Do not load all 11 million rows.** Start with a documented subset of 250,000 events.

**Interpretation:** each pitfall has a matching test or checklist item in `docs/12_QA_TEST_PLAN.md`.

## 7. Limitations and what comes next

- The events are simulated, and the signal is a hypothetical exotic-Higgs scenario.
- Classifier scores are not physical probabilities, and the benchmark does not give discovery significance.
- Results depend on the subset size, split, seed and tuning effort, so every run records them.
- Real model results will appear only in the mid-semester and end-semester notebooks, generated from saved artifacts.

**Sources:** see `docs/07_REFERENCES.md`.